In [12]:
import torch
import torch.nn as nn
from torchvision.models.vgg import vgg16

device = "cuda" if torch.cuda.is_available() else "cpu"

model = vgg16(weights= 'VGG16_Weights.DEFAULT') # pretrained -> weights
model

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

# 스킵커넥션

In [15]:
import torch
from torchvision.datasets.cifar import CIFAR10
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((32,32)), #이미지 사이즈 통일
    transforms.ToTensor(), #텐서로 변환
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) #정규화
])

train_dataset = CIFAR10(root = './', train=True, download=True, transform=transform)
test_dataset = CIFAR10(root = './', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
import torch
import torch.nn as nn

conv1 = nn.Conv2d(3,32, kernel_size=3,padding=1)
batch1 = nn.BatchNorm2d(32)
relu = nn.ReLU()
conv2 = nn.Conv2d(32,64, kernel_size=3,padding=1)
batch2 = nn.BatchNorm2d(64)


x = torch.randn(2,3,32,32)

x_= x #스킵커넥션을 위해 초기 입력을 저장
x = conv1(x)
x = batch1(x)
x = relu(x)
x = conv2(x)
x = batch2(x)

x.size(), x_.size()

(torch.Size([2, 64, 32, 32]), torch.Size([2, 3, 32, 32]))

In [ ]:
downsample = nn.Conv2d()

In [21]:
import torch
import torch.nn as nn

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, hidden_dim):
        super(BasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, hidden_dim, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(hidden_dim, out_channels, kernel_size=3, padding=1)
        self.batch1 = nn.BatchNorm2d(hidden_dim)
        self.batch2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()
        self.downsample = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    def forward(self, x):
       x = self.relu(self.batch1(self.conv1(x)))
       x = self.batch2(self.conv2(x))
       x += x_
       out = self.relu(x)
       print(out)
    

In [22]:
class ResNet(nn.Module):
    def __init__(self, class_num=10):
        super(ResNet, self).__init__()
        self.b1 = BasicBlock(3,64,32)
        self.b2 = BasicBlock(64,256,128)
        self.b3 = BasicBlock(256,256)

        self.pool = nn.AvgPool2d(2)
        # 분류기
        self.fc1 = nn.Linear(256*32*32, 2048)
        self.fc2 = nn.Linear(2048, 512)
        self.fc3 = nn.Linear(512, class_num)

        self.relu = nn.ReLu()

    def forward(self, x):
        x = self.pool(self.b1(x))
        x = self.pool(self.b2(x))
        x = self.pool(self.b3(x))
        x = torch.flatten(x, start_dim=1)

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        out = self.fc3(x)

        return out

In [26]:
from torch.optim import Adam
from tqdm import tqdm


#---- 2. 학습루프 ----
lr = 1e-3
optim = Adam(model.parameters(), lr=lr)
epochs = 1

for epoch in range(epochs):
    tqdm_obj = tqdm(train_loader,desc=f'epoch : {epoch+1}/{epochs}')
    for data, label in tqdm_obj:
        optim.zero_grad()
        preds = model(data.to(device))
        loss = nn.CrossEntropyLoss()(preds, label.to(device))
        loss.backward()
        optim.step()

        tqdm_obj.set_postfix(loss=loss.item())

epoch : 1/1:   0%|          | 0/1563 [00:00<?, ?it/s]

epoch : 1/1: 100%|██████████| 1563/1563 [23:06<00:00,  1.13it/s, loss=1.47] 


In [27]:
num_corr = 0
model.eval()

with torch.no_grad():
    for data, label in test_loader:
        output = model(data.to(device))
        preds = output.data.max(1)[1]
        corr = preds.eq(label.to(device).data).sum().item()
        num_corr += corr
    print(f'Accuracy: {num_corr / len(test_loader)}')

Accuracy: 22.7444089456869
